In [ ]:
import numpy as np
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, LeakyReLU, MaxPooling2D, Dropout, concatenate, UpSampling2D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import backend
import tensorflow as tf
import math
print(tf.__version__)

def high_freq_mae(y_true, y_pred):
    freq = tf.range(512,dtype=tf.float32)
    
    weights = 1.0 + (freq / 512.0) * 4.0 
    
    weights = tf.reshape(weights, (1, 512, 1 ,1))
    
    mae = tf.abs(y_true - y_pred)
    weighted_mae = mae * weights

    return tf.reduce_mean(weighted_mae)

#Unet network
def unet(pretrained_weights = None,input_size = (512,512,1)):
    #size filter input
    size_filter_in = 64
    #normal initialization of weights
    kernel_init = 'he_normal'
    activation_layer = None
    inputs = Input(input_size)
    conv1 = Conv2D(size_filter_in, 3, activation = activation_layer, padding = 'same', kernel_initializer = kernel_init)(inputs)
    conv1 = LeakyReLU()(conv1)
    conv1 = Conv2D(size_filter_in, 3, activation = activation_layer, padding = 'same', kernel_initializer = kernel_init)(conv1)
    conv1 = LeakyReLU()(conv1)
    pool1 = MaxPooling2D(pool_size=(2, 2))(conv1)

    conv2 = Conv2D(size_filter_in*2, 3, activation = activation_layer, padding = 'same', kernel_initializer = kernel_init)(pool1)
    conv2 = LeakyReLU()(conv2)
    conv2 = Conv2D(size_filter_in*2, 3, activation = activation_layer, padding = 'same', kernel_initializer = kernel_init)(conv2)
    conv2 = LeakyReLU()(conv2)
    pool2 = MaxPooling2D(pool_size=(2, 2))(conv2)

    conv3 = Conv2D(size_filter_in*4, 3, activation = activation_layer, padding = 'same', kernel_initializer = kernel_init)(pool2)
    conv3 = LeakyReLU()(conv3)
    conv3 = Conv2D(size_filter_in*4, 3, activation = activation_layer, padding = 'same', kernel_initializer = kernel_init)(conv3)
    conv3 = LeakyReLU()(conv3)
    pool3 = MaxPooling2D(pool_size=(2, 2))(conv3)

    conv4 = Conv2D(size_filter_in*8, 3, activation = activation_layer, padding = 'same', kernel_initializer = kernel_init)(pool3)
    conv4 = LeakyReLU()(conv4)
    conv4 = Conv2D(size_filter_in*8, 3, activation = activation_layer, padding = 'same', kernel_initializer = kernel_init)(conv4)
    conv4 = LeakyReLU()(conv4)
    drop4 = Dropout(0.5)(conv4)
    pool4 = MaxPooling2D(pool_size=(2, 2))(drop4)

    conv5 = Conv2D(size_filter_in*16, 3, activation = activation_layer, padding = 'same', kernel_initializer = kernel_init)(pool4)
    conv5 = LeakyReLU()(conv5)
    conv5 = Conv2D(size_filter_in*16, 3, activation = activation_layer, padding = 'same', kernel_initializer = kernel_init)(conv5)
    conv5 = LeakyReLU()(conv5)
    drop5 = Dropout(0.5)(conv5)

    up6 = Conv2D(size_filter_in*8, 2, activation = activation_layer, padding = 'same', kernel_initializer = kernel_init)(UpSampling2D(size = (2,2))(drop5))
    up6 = LeakyReLU()(up6)
    merge6 = concatenate([drop4,up6], axis = 3)
    conv6 = Conv2D(size_filter_in*8, 3, activation = activation_layer, padding = 'same', kernel_initializer = kernel_init)(merge6)
    conv6 = LeakyReLU()(conv6)
    conv6 = Conv2D(size_filter_in*8, 3, activation = activation_layer, padding = 'same', kernel_initializer = kernel_init)(conv6)
    conv6 = LeakyReLU()(conv6)

    up7 = Conv2D(size_filter_in*4, 2, activation = activation_layer, padding = 'same', kernel_initializer = kernel_init)(UpSampling2D(size = (2,2))(conv6))
    up7 = LeakyReLU()(up7)
    merge7 = concatenate([conv3,up7], axis = 3)
    conv7 = Conv2D(size_filter_in*4, 3, activation = activation_layer, padding = 'same', kernel_initializer = kernel_init)(merge7)
    conv7 = LeakyReLU()(conv7)
    conv7 = Conv2D(size_filter_in*4, 3, activation = activation_layer, padding = 'same', kernel_initializer = kernel_init)(conv7)
    conv7 = LeakyReLU()(conv7)

    up8 = Conv2D(size_filter_in*2, 2, activation = activation_layer, padding = 'same', kernel_initializer = kernel_init)(UpSampling2D(size = (2,2))(conv7))
    up8 = LeakyReLU()(up8)
    merge8 = concatenate([conv2,up8], axis = 3)
    conv8 = Conv2D(size_filter_in*2, 3, activation = activation_layer, padding = 'same', kernel_initializer = kernel_init)(merge8)
    conv8 = LeakyReLU()(conv8)
    conv8 = Conv2D(size_filter_in*2, 3, activation = activation_layer, padding = 'same', kernel_initializer = kernel_init)(conv8)
    conv8 = LeakyReLU()(conv8)

    up9 = Conv2D(size_filter_in, 2, activation = activation_layer, padding = 'same', kernel_initializer = kernel_init)(UpSampling2D(size = (2,2))(conv8))
    up9 = LeakyReLU()(up9)
    merge9 = concatenate([conv1,up9], axis = 3)
    conv9 = Conv2D(size_filter_in, 3, activation = activation_layer, padding = 'same', kernel_initializer = kernel_init)(merge9)
    conv9 = LeakyReLU()(conv9)
    conv9 = Conv2D(size_filter_in, 3, activation = activation_layer, padding = 'same', kernel_initializer = kernel_init)(conv9)
    conv9 = LeakyReLU()(conv9)

    conv10 = Conv2D(1, 1, activation = 'sigmoid')(conv9)

    model = Model(inputs,conv10)

    model.compile(optimizer = 'adam', loss = high_freq_mae, metrics = ['mae'])

    #model.summary()

    if(pretrained_weights):
    	model.load_weights(pretrained_weights)

    return model


In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')
root_path = 'gdrive/My Drive/ML_audio/'

In [ ]:
import librosa
import os

currentdir = os.getcwd()
datadir = f"{currentdir}/data"
# file_no_dolby = "ELO_1_raw.wav"
# file_dolby    = "ELO_1_clean.wav"
# file_no_dolby = "ELO_1-2.wav"
# file_dolby    = "ELO_1_clean-3.wav"
file_no_dolby = "AUDIO_all_2.wav"
file_dolby    = "AUDIO_all_clean_2.wav"

sr = 44100
frame_length = 262144
hop_length_frame = 131072

dim_square_spec = 512
n_fft = 1023
hop_length_fft = 512


In [ ]:
y_no_dolby, sr1 = librosa.load(
    os.path.join(root_path, file_no_dolby),
    sr=sr,
    mono=True
)

y_dolby, sr2 = librosa.load(
    os.path.join(root_path, file_dolby),
    sr=sr,
    mono=True
)

print("No Dolby length:", y_no_dolby.shape[0])
print("Dolby length   :", y_dolby.shape[0])


In [ ]:
class SlicedAudioGenerator(tf.keras.utils.Sequence):
    def __init__(self, noisy_data, clean_data, batch_size, frame_length, n_fft, hop_length, 
                 target_shape, shuffle=True):
        super().__init__()
        self.noisy_data = noisy_data
        self.clean_data = clean_data
        self.batch_size = batch_size
        self.frame_length = frame_length
        self.n_fft = n_fft
        self.hop_length = hop_length
        self.target_shape = target_shape
        self.shuffle = shuffle

        self.total_samples = len(self.noisy_data)
        self.n_frames = self.total_samples // self.frame_length
        self.indices = np.arange(self.n_frames)
        
    def __len__(self):
        return int(math.floor(self.n_frames / self.batch_size))
    
    def get_mag(self, audio):
        stft = librosa.stft(audio, n_fft=self.n_fft, hop_length=self.hop_length)
        mag, _ = librosa.magphase(stft)
        
        if mag.shape[1] > self.target_shape[1]:
            mag = mag[:, :self.target_shape[1]]
        elif mag.shape[1] > self.target_shape[1]:
            mag = np.pad(mag, ((0,0), (0, (self.target_shape - mag.shape[1]))))
            
        return mag
    
    def __getitem__(self, index):
        batch_indices = self.indices[index * self.batch_size : (index + 1) * self.batch_size]
        X = np.empty((self.batch_size, *self.target_shape, 1))
        Y = np.empty((self.batch_size, *self.target_shape, 1))
        
        for i, idx in enumerate(batch_indices):
            start = idx * self.frame_length
            end = start + self.frame_length
            
            audio_x = self.noisy_data[start: end]
            audio_y = self.clean_data[start: end]
            
            mag_x = self.get_mag(audio_x)
            mag_y = self.get_mag(audio_y)
            
            mask = np.clip(mag_y / (mag_x + 1e-10))
            
            mag_x_db = librosa.amplitude_to_db(mag_x, ref=np.max)
            mag_x_db_scaled = (mag_x_db / 40.1) + 1.0
            
            X[i, :, :, 0] = mag_x_db_scaled
            Y[i, :, :, 0] = mask
            
        return X, Y
    
    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)

In [ ]:
split_idx = int(len(y_no_dolby) * 0.9)

frame_len = 262144
split_idx = (split_idx // frame_len) * frame_len

X_train = y_no_dolby[:split_idx]
Y_train = y_dolby[:split_idx]

X_val = y_no_dolby[split_idx:]
Y_val = y_dolby[split_idx:]

train_gen = SlicedAudioGenerator(X_train, Y_train, 16, frame_len, n_fft, hop_length_fft, (512, 512))
val_gen = SlicedAudioGenerator(X_val, Y_val, 16, frame_len, n_fft, hop_length_fft, (512, 512), False)

In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint

generator_nn = unet()
generator_nn.compile(optimizer=Adam(learning_rate=1e-4), loss=high_freq_mae)
generator_nn.summary()
checkpoint = ModelCheckpoint('model_best.h5', monitor='val_loss', save_best_only=True)
history = generator_nn.fit(train_gen, validation_data=val_gen, epochs=50, batch_size=80, callbacks=[checkpoint])


In [ ]:
from matplotlib import pyplot as plt
loss = history.history['loss']
val_loss = history.history['val_loss']
epochs = range(1, len(loss) + 1)

plt.plot(epochs, loss, label='Training loss')
plt.plot(epochs, val_loss, label='Validation loss')
plt.yscale('log')
plt.title('Training and validation loss')
plt.legend()
plt.show()

In [ ]:
model_json = generator_nn.to_json()
with open(root_path + "mod_unet_last_weights_v16.json", "w") as json_file:
    json_file.write(model_json)

generator_nn.save_weights(root_path + "mod_unet_last_weights_v16.weights.h5")
